# EDA - Bike Sharing Demand
UE24CS352A ML Mini-Project. Goal: predict hourly `count`, metric RMSLE. Findings feed `src/features.py`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
train = pd.read_csv("../data/train.csv", parse_dates=["datetime"])
test = pd.read_csv("../data/test.csv", parse_dates=["datetime"])
print(train.shape, test.shape)
train.head()

## 1. Structure, types, missing values

In [ ]:
display(train.dtypes.to_frame("dtype").T)
print("Missing (train):", train.isna().sum().sum(), "| Missing (test):", test.isna().sum().sum())
print("Duplicated timestamps:", train.datetime.duplicated().sum())
print("Train dates:", train.datetime.min(), "->", train.datetime.max())
print("Test dates :", test.datetime.min(), "->", test.datetime.max())
print("Train days-of-month:", sorted(train.datetime.dt.day.unique())[:3], "...", train.datetime.dt.day.max())
print("Test days-of-month :", test.datetime.dt.day.min(), "...", test.datetime.dt.day.max())
train.describe().T

**Note:** train = days 1-19 of each month, test = days 20+ of the same months. So this is interpolation across time, not pure forecasting.

## 2. Target distribution -> why log1p

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(train["count"], bins=50, ax=ax[0]); ax[0].set_title("count (right-skewed)")
sns.histplot(np.log1p(train["count"]), bins=50, ax=ax[1]); ax[1].set_title("log1p(count)")
plt.tight_layout(); plt.show()
print("skew raw:", round(train["count"].skew(), 2), "| skew log1p:", round(np.log1p(train["count"]).skew(), 2))

RMSLE = RMSE on log1p, so training on `log1p(count)` directly optimises the metric and tames the skew.

In [ ]:
d = train.copy()
d["hour"] = d.datetime.dt.hour; d["month"] = d.datetime.dt.month
d["dow"] = d.datetime.dt.dayofweek; d["year"] = d.datetime.dt.year
d["daytype"] = np.where(d.workingday == 1, "working day", "weekend/holiday")

## 3. Hour of day (the strongest signal)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=d, x="hour", y="count", hue="daytype", ax=ax[0]); ax[0].set_title("Mean count by hour")
sns.lineplot(data=d.melt(id_vars="hour", value_vars=["casual","registered"]), x="hour", y="value", hue="variable", ax=ax[1])
ax[1].set_title("Casual vs registered by hour"); plt.tight_layout(); plt.show()

Working days: twin commute peaks (~8am, ~5-6pm). Weekends: one broad midday hump. Registered users drive the commute peaks; casual users the weekend hump -> motivates `peak_weekday` / `peak_weekend` flags.

## 4. Month, season, year, day of week

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
sns.barplot(data=d, x="month", y="count", ax=ax[0]); ax[0].set_title("By month")
sns.barplot(data=d, x="year", y="count", ax=ax[1]); ax[1].set_title("By year (growth)")
sns.barplot(data=d, x="dow", y="count", ax=ax[2]); ax[2].set_title("By day of week (0=Mon)")
plt.tight_layout(); plt.show()
print(d.groupby("season")["count"].mean().round(1).to_dict())

## 5. Weather and temperature

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
sns.barplot(data=d, x="weather", y="count", ax=ax[0]); ax[0].set_title("By weather (1=clear .. 4=storm)")
ax[1].scatter(d.temp, d["count"], s=3, alpha=.3); ax[1].set_title("temp vs count (not a straight line)")
ax[1].set_xlabel("temp")
ax[2].scatter(d.humidity, d["count"], s=3, alpha=.3); ax[2].set_title("humidity vs count"); ax[2].set_xlabel("humidity")
plt.tight_layout(); plt.show()
print("weather counts:", d.weather.value_counts().sort_index().to_dict())

Demand rises with temperature but flattens/drops when hot -> **temperature buckets** help linear models. Weather 4 is very rare.

## 6. Holiday and working day

In [ ]:
print(d.groupby("holiday")["count"].agg(["mean", "count"]).round(1))
print(d.groupby("workingday")["count"].agg(["mean", "count"]).round(1))

Holidays are only a small share of rows and, after accounting for hour + workingday, add little -> candidate to **drop** (tested in ablation).

## 7. Correlations / collinearity

In [ ]:
num = ["temp", "atemp", "humidity", "windspeed", "casual", "registered", "count"]
plt.figure(figsize=(6.5, 5)); sns.heatmap(train[num].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0); plt.show()

`temp` and `atemp` are ~0.98 correlated -> collinearity for linear models; ablation tests dropping `temp`.

## 8. Outliers and oddities

In [ ]:
print("windspeed == 0:", (train.windspeed == 0).mean().round(3), "(likely missing values coded as 0)")
print(d.groupby("hour")["count"].quantile([.5, .99]).unstack().round(0).T)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.boxplot(data=d, x="hour", y="count", ax=ax[0]); ax[0].set_title("count by hour")
sns.histplot(train.windspeed, bins=40, ax=ax[1]); ax[1].set_title("windspeed")
plt.tight_layout(); plt.show()

## 9. Summary -> feature decisions
- Train on `log1p(count)`, evaluate RMSLE.
- Parse datetime -> hour, month, day of week, year.
- Hour is dominant; add weekday/weekend peak flags.
- One-hot categoricals + temperature buckets for linear models.
- `temp`/`atemp` collinear; `holiday` weak -> ablation decides.
- Casual vs registered behave differently by hour -> separate-model experiment.